# Career Compass SG — Exploratory Data Analysis

**Module 1 Group 7 · Singapore Jobs Analytics · Topic: Career Recommendations**

This notebook is the evidence behind the dashboard. It follows the order taught in
Lessons 1.8 (EDA Basic), 1.9 (EDA Advanced) and 1.10 (Data Visualisation & Storytelling):

| Part | Question | Lesson |
|---|---|---|
| 0 | Load a sample, not the whole file | 1.8 Getting Started |
| 1 | How big is this, and what is in it? | 1.8 Part 1 — Descriptive statistics |
| 2 | Can I trust it? | 1.8 Part 2 — Missing data, duplicates, impossible values |
| 3 | What should I build from it? | 1.8 Part 3 — Transformation & feature engineering |
| 4 | What is the pattern? | 1.9 — Grouping and aggregation |
| 5 | What is the message? | 1.10 — Charts that state a finding |
| 6 | From analysis to product | 1.9 — Weighted scores |
| 7 | How is the market structured? | 1.9 — Grouping, benchmarking; 1.10 — Charts |

> **The one rule from Lesson 1.10 we followed throughout:**
> *the chart is chosen by the message, not by taste.* Every chart below has its
> finding written above it in words first.

## Setup

`data_utils.py` (one folder up) holds the production code. We import the **same functions the
dashboard uses**, so anything demonstrated here is literally what runs in `app.py`: the notebook
and the app cannot drift apart.

**To run:** put the course file at `data/SGJobData.csv` (next to `app.py`), then *Run All*.
Charts are saved to `figures/`.

In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# The project folder holds app.py, data_utils.py and data/. Works whether the
# notebook is started from notebooks/ or from the project folder.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))

import data_utils as du
from data_utils import add_features, build_analysis_tables, clean_chunk, explode_categories, parse_categories

config = SimpleNamespace(
    RAW_CSV=PROJECT / "data" / "SGJobData.csv",
    FIGURES_DIR=PROJECT / "figures",
    SAMPLE_ROWS=50_000,
    SENIORITY_ORDER=du.SENIORITY_ORDER,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 110

RAW = config.RAW_CSV
FIGS = config.FIGURES_DIR
FIGS.mkdir(parents=True, exist_ok=True)
print("Raw file:", RAW.name, f"({RAW.stat().st_size / 1e6:,.0f} MB)")

---
## Part 0 — Taste a spoonful before cooking the pot

The brief is explicit about this and it is good advice: the file is ~1M rows and
273 MB. Every decision below was designed on a 50,000-row sample and only then run
over the full file.

In [ ]:
sample = pd.read_csv(RAW, nrows=config.SAMPLE_ROWS)

# The three first EDA moves from the brief
print("1. shape:", sample.shape)
print()
sample.info()

In [ ]:
# 3. Pick one numeric column — what is typical, and what is extreme?
sample["salary_minimum"].describe()

**First red flag, thirty seconds in.** The minimum monthly salary is **0** and the
maximum is **180,000**. Neither is a wage: one is "not disclosed" and the other is an
annual figure typed into a monthly field. We come back to this in Part 2.

---
## Part 1 — How big is this, and what is in it?

### 1.1 The five-move first look (Lesson 1.8 §1.1)

In [ ]:
sample.head(3)

In [ ]:
# Move 4: just the types. `object` means text (or mixed).
sample.dtypes.value_counts()

In [ ]:
# Move 5: the numbers. Read the min and max rows first - that is where impossible
# values hide.
sample.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(1)

### 1.2 The categorical columns

`.value_counts()` on the text columns tells us what we are actually dealing with.

In [ ]:
for col in ["employmentTypes", "positionLevels", "status_jobStatus", "salary_type"]:
    print(f"--- {col} ---")
    print(sample[col].value_counts().head(10))
    print()

Three things worth writing down:

1. **`salary_type` is 100% `Monthly`.** No currency or period conversion is needed —
   one less way to get the analysis wrong.
2. **`positionLevels` has 9 values.** Nine is too many for a person to reason about;
   we collapse them to 5 seniority bands in Part 3.
3. **`status_jobStatus` is dominated by one value.** Worth remembering when we later
   ask why so many postings have zero applications.

### 1.3 The `categories` column is not a category

This is the one column the brief warns about, and it is the reason the whole project
needs two tables instead of one.

In [ ]:
print(sample["categories"].iloc[0])
print()
print("parsed ->", parse_categories(sample["categories"].iloc[0]))

In [ ]:
cat_counts = sample["categories"].dropna().map(parse_categories)
print(f"A job belongs to {cat_counts.map(len).mean():.2f} categories on average")
print(f"Maximum categories on one job: {cat_counts.map(len).max()}")
print(f"Distinct categories in the sample: {len(set(c for lst in cat_counts for c in lst))}")

> **Design consequence.** Counting rows answers *"how many postings are there"*.
> Counting rows **after exploding categories** answers *"how many IT jobs are there"*.
> They are different questions, so the pipeline produces two tables — a wide one
> (`jobs_clean.parquet`, one row per posting) and a long one
> (`jobs_by_category.parquet`, one row per posting × category). Category shares
> therefore overlap and do not sum to 100%, which we say on the dashboard.

---
## Part 2 — Can I trust it? (Lesson 1.8 Part 2)

### 2.1 Missing data

In [ ]:
missing = (sample.isna().mean() * 100).round(2).sort_values(ascending=False)
missing[missing > 0]

Two very different kinds of missing:

* **`occupationId` — 100% empty.** Verified across all 1,048,585 rows, not just the
  sample. A column that is always empty cannot inform anything, so it is dropped.
* **Everything else — 0.14%.** Notice they are all *the same* 0.14%. That is the
  signature of broken export lines rather than of individually missing fields.

In [ ]:
# Are the ~0.14% of gaps the same rows? If so we can drop whole rows, not columns.
broken = sample[sample["metadata_jobPostId"].isna()]
print(f"Rows with no job id: {len(broken)}")
print("Of those, how many are also missing title/company/dates?")
print(broken[["title", "postedCompany_name", "metadata_originalPostingDate"]].isna().sum())

Confirmed: they are entirely empty rows. **Decision — drop the rows, keep the columns.**
Over the full file this removes 3,988 rows (0.38%).

### 2.2 Duplicates

The brief hints that the same job is re-posted, so we expected duplicate ids. We checked
rather than assumed.

In [ ]:
ids = sample["metadata_jobPostId"]
print("duplicated() including blanks :", ids.duplicated().sum())
print("duplicated() excluding blanks :", ids.dropna().duplicated().sum())

> **A trap worth naming in the report.** `.duplicated()` counts repeated *blanks* as
> duplicates. The first number looks like a duplicate problem; the second shows there
> is none. Across the full file there are **zero** genuine duplicate job ids — the
> repeated-posting signal lives in `metadata_repostCount` instead, not in duplicate rows.
> We still de-duplicate defensively in the pipeline (`keep="last"`), it simply removes nothing.

### 2.3 Impossible values (Lesson 1.8 §2.3)

This is where most of the cleaning judgement sits, so each rule is stated with its reason.

In [ ]:
s = sample
print(f"salary_minimum == 0        : {(s.salary_minimum == 0).sum():>6,}")
print(f"salary_minimum < 500 (>0)  : {((s.salary_minimum < 500) & (s.salary_minimum > 0)).sum():>6,}")
print(f"salary_maximum > 60,000    : {(s.salary_maximum > 60_000).sum():>6,}")
print(f"salary_minimum > maximum   : {(s.salary_minimum > s.salary_maximum).sum():>6,}")
print(f"minimumYearsExperience > 40: {(s.minimumYearsExperience > 40).sum():>6,}")

In [ ]:
# What does a "180,000 per month" posting actually look like?
s.loc[s.salary_maximum > 60_000, ["title", "salary_minimum", "salary_maximum", "positionLevels"]].head(5)

**The four salary rules, and why each one is what it is:**

| Rule | What we do | Why not something else |
|---|---|---|
| `salary == 0` | set to missing | Zero is "not disclosed", not "this job pays nothing". Treating it as a number would drag every average down. |
| `min > max` | swap the two values | The fields were entered the wrong way round. The information is there; dropping the row would throw away a real posting. |
| `min < $500` or `max > $60,000` | blank the salary, **keep the row** | These are annual figures in a monthly field. We still want the posting to count as demand — we just cannot trust its pay. |
| top/bottom 1% | **cap**, do not delete | A thin tail of genuine executive packages drags the mean. Capping keeps the row (so demand counts stay right) while stopping ~19,600 rows from steering every average. |

The consistent principle: **blank the untrustworthy field, keep the trustworthy row.**
A posting with a broken salary still tells us a job exists.

In [ ]:
# Run the real cleaning function on the sample and read its audit counters.
cleaned_sample, stats = clean_chunk(sample.copy())
pd.Series(stats)

### 2.4 The finding that changed the whole design

Every project has one moment where the data tells you something you did not ask.
Here it is. We plotted engagement (views, applications) over time purely as a sanity
check — and found the dataset is **two stitched extracts**.

In [ ]:
probe = pd.read_csv(
    RAW,
    usecols=[
        "metadata_originalPostingDate",
        "metadata_totalNumberOfView",
        "metadata_totalNumberJobApplication",
    ],
)
probe["month"] = pd.to_datetime(probe["metadata_originalPostingDate"]).dt.to_period("M")

engagement = probe.groupby("month").agg(
    postings=("metadata_totalNumberOfView", "size"),
    mean_views=("metadata_totalNumberOfView", "mean"),
    zero_application_pct=("metadata_totalNumberJobApplication", lambda x: (x == 0).mean() * 100),
).round(1)
engagement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
x = engagement.index.to_timestamp()
cut = pd.Timestamp("2023-07-01")

axes[0].plot(x, engagement["mean_views"], color="#c0392b", linewidth=2.2)
axes[0].axvline(cut, color="#2c3e50", linestyle="--", linewidth=1.4)
axes[0].set_title("Mean views per posting collapses after Jun 2023")
axes[0].set_ylabel("Mean views")
axes[0].set_ylim(0, None)

axes[1].plot(x, engagement["zero_application_pct"], color="#c0392b", linewidth=2.2)
axes[1].axvline(cut, color="#2c3e50", linestyle="--", linewidth=1.4)
axes[1].set_title("...and most postings show zero applications")
axes[1].set_ylabel("% of postings with 0 applications")
axes[1].set_ylim(0, 100)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", rotation=0)

fig.suptitle(
    "The engagement counters are only real for postings up to Jun 2023",
    fontsize=13, y=1.04, x=0.02, ha="left",
)
fig.tight_layout()
fig.savefig(FIGS / "eda_engagement_break.png", dpi=150, bbox_inches="tight")
plt.show()

> ### The single most important finding in this project
>
> Mean views per posting fall from **~111 to ~5** and the share of postings with zero
> applications jumps from **18% to 79%**, exactly at the Jun/Jul 2023 boundary.
>
> This is not a collapse in interest — it is a **collection artefact**. Postings from
> 2023-07 onward were captured at or near posting time, so their counters never
> accumulated.
>
> **What we did about it.** Demand and salary are trustworthy across the whole file and
> use all 1.04M rows. Competition (applications per vacancy) is computed **only** on
> postings up to 2023-06-30 — 203,702 of them — and is labelled as such everywhere it
> appears. Had we ignored this, every career track would have looked uncontested, and the
> recommender would have confidently pointed people at the most crowded markets in Singapore.

In [ ]:
# The same check for posting VOLUME - a second, smaller artefact.
volume = probe.groupby("month").size()
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(volume.index.to_timestamp(), volume.values, color="#2c3e50", linewidth=2)
ax.axvspan(
    volume.index.to_timestamp().min(), pd.Timestamp("2023-05-01"),
    color="#8c9196", alpha=0.15,
)
ax.text(pd.Timestamp("2022-11-01"), volume.max() * 0.72, "partial\ncollection",
        color="#5d6d7e", fontsize=10)
ax.set_ylim(0, None)
ax.set_ylabel("Postings per month")
ax.set_title("Volume is only complete from May 2023 onward", fontsize=13, loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGS / "eda_volume_ramp.png", dpi=150, bbox_inches="tight")
plt.show()

**Second design consequence.** Trend charts and all growth calculations start at
**May 2023**. A "growth" number that included the ramp would be measuring our data
collection, not Singapore's hiring.

---
## Part 3 — Feature engineering (Lesson 1.8 Part 3)

Raw columns describe a *posting*. A job seeker needs columns that describe an
*opportunity*. These are the features we added and the question each one answers.

| Feature | The question it answers |
|---|---|
| `seniority` | 9 official levels → 5 bands a person actually thinks in |
| `salary_band`, `experience_band` | "which bracket am I in?" |
| `applications_per_vacancy` | **"how many people am I up against for each seat?"** |
| `apply_rate` | of the people who saw it, how many acted? |
| `is_hard_to_fill` | the employer could not fill this — that is an opening |
| `is_entry_friendly` | can I apply today, with what I have? |
| `title_clean` | count *roles*, not adverts |
| `skill` (long table) | which terms in a title carry a pay premium |

In [ ]:
featured = add_features(cleaned_sample)
featured[["title", "title_clean"]].dropna().head(8)

**Why `title_clean` matters.** Singapore job titles carry a lot of marketing:
`"Urgent Hiring!!! Business Development Manager (MES, Pre-sales) - Up to $9K"`.
Counting raw titles counts *adverts*; counting cleaned titles counts *roles*.
Without this step the "most common job title" list is mostly noise.

In [ ]:
print("Raw distinct titles     :", featured["title"].nunique())
print("Cleaned distinct titles :", featured["title_clean"].nunique())
print()
print("Top 8 roles after cleaning:")
print(featured["title_clean"].value_counts().head(8))

---
## Part 4 — What is the pattern? (Lesson 1.9)

From here we switch from the sample to the **full cleaned dataset**, because the questions now
need every row. `build_analysis_tables()` reads the whole CSV once, cleans it with the same rules
as Part 2, and returns every table the rest of the notebook uses. Expect it to take a minute.

In [ ]:
raw = pd.read_csv(RAW, low_memory=False)
jobs, by_cat, categories, skills, tracks, quality = build_analysis_tables(raw)
del raw  # free ~1 GB of memory; everything below uses the cleaned tables

print(f"postings          : {len(jobs):,}")
print(f"job x category    : {len(by_cat):,}")
print(f"career tracks     : {len(tracks):,}")
print(f"rows read from CSV: {quality['rows_read']:,}")
print(f"rows kept         : {quality['rows_after_cleaning']:,} "
      f"({quality['rows_after_cleaning'] / quality['rows_read']:.2%})")
pd.Series(quality, name="rows affected").to_frame()

In [ ]:
# The market in one table
pd.Series({
    "Postings": f"{len(jobs):,}",
    "Vacancies (seats)": f"{int(jobs['vacancies'].sum()):,}",
    "Categories": jobs.shape[0] and by_cat["category"].nunique(),
    "Posting organisations": f"{jobs['company'].nunique():,}",
    "Median salary": f"${jobs['avg_salary'].median():,.0f}",
    "25th / 75th percentile": f"${jobs['avg_salary'].quantile(.25):,.0f} / ${jobs['avg_salary'].quantile(.75):,.0f}",
    "Salary disclosed": f"{jobs['avg_salary'].notna().mean():.1%}",
    "Entry-friendly (<=1 yr)": f"{jobs['is_entry_friendly'].mean():.1%}",
    "Reposted at least once": f"{jobs['is_hard_to_fill'].mean():.1%}",
}, name="value").to_frame()

In [ ]:
# Salary by seniority - the ladder
jobs.groupby("seniority", observed=True)["avg_salary"].agg(
    postings="size", median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75)
).reindex(config.SENIORITY_ORDER).round(0)

In [ ]:
# The 10 largest categories, with everything a job seeker would ask
categories.head(10)[[
    "category", "postings", "median_salary", "apps_per_vacancy",
    "entry_friendly_share", "hard_to_fill_share", "growth_pct_6m",
]].round(1)

### 4.1 Three patterns worth stating in words

**(a) Size and pay are almost unrelated.** Admin/Secretarial is the 3rd largest
category (117,854 postings) and one of the worst paid ($2,900). Information Technology
is the largest *and* the best paid ($6,500). "Where the jobs are" and "where the money
is" are genuinely different questions — which is precisely why the recommender scores
both instead of ranking on one.

**(b) Competition varies by an order of magnitude.** Personal Care / Beauty sees
**0.7 applicants per seat**; Social Services sees **11.3**. Two categories with the same
number of openings can offer wildly different odds, and nothing on a job board tells
you this.

**(c) Hard-to-fill means underpaid, not elite.** Reposted jobs pay a median of
**$3,700** against **$3,850** for jobs filled first time. It is a negotiation lever,
not a prestige signal.

In [ ]:
print("Competition spread across sizeable categories (>= 5,000 postings):")
big = categories[categories.postings >= 5000]
print(big.nsmallest(5, "apps_per_vacancy")[["category", "postings", "median_salary", "apps_per_vacancy"]].round(2).to_string(index=False))
print("   ...")
print(big.nlargest(5, "apps_per_vacancy")[["category", "postings", "median_salary", "apps_per_vacancy"]].round(2).to_string(index=False))
print()
print("Median salary, reposted vs filled first time:")
print(jobs.groupby("is_hard_to_fill")["avg_salary"].agg(postings="size", median="median"))

---
## Part 5 — What is the message? (Lesson 1.10)

Four charts. Each one has its finding written above it as a sentence, then the chart
is built to deliver that sentence — colour used as an argument, everything else grey.

### Chart 1 — Where the jobs are is not where the money is

*Message: the biggest categories are not the best-paying ones, with one exception.*

In [ ]:
top = categories.head(12).sort_values("postings")
market_median = jobs["avg_salary"].median()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)

# Left: volume. One orange bar carries the point; the rest are context.
colours = ["#e07b39" if c == "Information Technology" else "#b8c4d0" for c in top["category"]]
axes[0].barh(top["category"], top["postings"], color=colours)
axes[0].set_title("Postings", fontsize=12, loc="left")
axes[0].set_xlabel("Job postings")

# Right: pay, same row order, same highlight.
axes[1].barh(top["category"], top["median_salary"], color=colours)
axes[1].axvline(market_median, color="#2c3e50", linestyle="--", linewidth=1.3)
axes[1].text(market_median * 1.02, -0.4, f"market median ${market_median:,.0f}",
             color="#2c3e50", fontsize=9)
axes[1].set_title("Median monthly salary", fontsize=12, loc="left")
axes[1].set_xlabel("SGD")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Information Technology is the only category that is both the biggest and the best paid",
    fontsize=13.5, x=0.01, ha="left", y=1.02,
)
fig.tight_layout()
fig.savefig(FIGS / "eda_volume_vs_pay.png", dpi=150, bbox_inches="tight")
plt.show()

### Chart 2 — What an average hides

*Message: the salary distribution is right-skewed, so the mean describes almost nobody.*
This is the Lesson 1.10 §3.1 lesson applied to our own data.

In [ ]:
salaries = jobs["avg_salary"].dropna()
mean_v, median_v = salaries.mean(), salaries.median()

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(salaries, bins=60, color="#2c3e50", ax=ax)
ax.axvline(median_v, color="#e07b39", linewidth=2.2)
ax.axvline(mean_v, color="#8c9196", linewidth=2, linestyle="--")
ax.text(median_v * 0.98, ax.get_ylim()[1] * 0.92, f"median ${median_v:,.0f}",
        color="#e07b39", ha="right", fontsize=11, fontweight="bold")
ax.text(mean_v * 1.03, ax.get_ylim()[1] * 0.92, f"mean ${mean_v:,.0f}",
        color="#5d6d7e", fontsize=11)
ax.set_xlabel("Average monthly salary (SGD)")
ax.set_ylabel("Postings")
ax.set_title(
    f"Half of Singapore's postings pay under ${median_v:,.0f} a month",
    fontsize=13, loc="left",
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGS / "eda_salary_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"skew: {salaries.skew():.2f}  |  mean - median = ${mean_v - median_v:,.0f}")

### Chart 3 — The opportunity map

*Message: volume alone is a bad guide; the categories worth targeting are the ones with
volume AND low competition.* This chart is the analytical core of the dashboard's
Demand vs Competition page.

In [ ]:
plot = categories[(categories.postings >= 2000) & categories.apps_per_vacancy.notna()]
x_split = plot["apps_per_vacancy"].median()
y_split = plot["postings"].median()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    plot["apps_per_vacancy"], plot["postings"],
    s=plot["median_salary"] / 12, alpha=0.6,
    color="#2c3e50", edgecolor="white", linewidth=1.2,
)
ax.axvline(x_split, color="#b8c4d0", linestyle=":", linewidth=1.4)
ax.axhline(y_split, color="#b8c4d0", linestyle=":", linewidth=1.4)
ax.set_yscale("log")

# Label sparingly: the 6 biggest categories in the best-odds quadrant carry the
# message, plus the 3 biggest overall for orientation. Labelling every point
# produced unreadable collisions.
best_odds = plot[(plot.postings > y_split) & (plot.apps_per_vacancy < x_split)]
to_label = set(best_odds.nlargest(6, "postings")["category"])
context = set(plot.nlargest(3, "postings")["category"]) - to_label

for i, (_, r) in enumerate(plot.iterrows()):
    if r["category"] in to_label:
        colour, weight = "#1e7a5f", "bold"
    elif r["category"] in context:
        colour, weight = "#5d6d7e", "normal"
    else:
        continue
    offset = (8, 5) if i % 2 == 0 else (8, -11)
    ax.annotate(r["category"], (r["apps_per_vacancy"], r["postings"]),
                textcoords="offset points", xytext=offset,
                fontsize=9, color=colour, fontweight=weight)

ax.text(0.02, 0.96, "BEST ODDS\nmany jobs, few rivals", transform=ax.transAxes,
        color="#1e7a5f", fontsize=10.5, va="top", fontweight="bold")
ax.text(0.98, 0.96, "CROWDED\nmany jobs, many rivals", transform=ax.transAxes,
        color="#c0392b", fontsize=10.5, va="top", ha="right", fontweight="bold")

ax.set_xlabel("Applicants per seat  (postings up to Jun 2023)")
ax.set_ylabel("Job postings (log scale)")
ax.set_title(
    "Customer Service, Sales / Retail and F&B offer real volume without the queue",
    fontsize=13.5, loc="left",
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGS / "eda_opportunity_map.png", dpi=150, bbox_inches="tight")
plt.show()

> **Why the y-axis is logarithmic.** Category sizes span two orders of magnitude
> (517 to 140,866 postings). On a linear axis the smaller half of the market collapses
> onto the floor and becomes unreadable. A log axis is the honest choice here — and we
> label it as such, because an unlabelled log axis is one of the easiest ways to mislead.

### Chart 4 — What skills are worth

*Message: the pay gap between the best- and worst-paying title keywords is a factor of
four, and it is the clearest investment signal in the dataset.*

In [ ]:
ranked = skills[skills.postings >= 1500].sort_values("salary_premium_pct")
# Show the 8 lowest and 8 highest premiums (or every keyword, if there are 16 or fewer).
ends = ranked if len(ranked) <= 16 else pd.concat([ranked.head(8), ranked.tail(8)])
colours = ["#c0392b" if v < 0 else "#1e7a5f" for v in ends["salary_premium_pct"]]

fig, ax = plt.subplots(figsize=(9.5, 6))
ax.barh(ends["skill"], ends["salary_premium_pct"], color=colours)
ax.axvline(0, color="#2c3e50", linewidth=1.2)
for y, (v, n) in enumerate(zip(ends["salary_premium_pct"], ends["postings"])):
    ax.text(v + (2 if v > 0 else -2), y, f"{v:+.0f}%",
            va="center", ha="left" if v > 0 else "right", fontsize=9.5, color="#3d4852")
ax.set_xlabel(f"Median salary vs the market median of ${jobs['avg_salary'].median():,.0f} (%)")
ax.set_xlim(-70, 155)

best, worst = ends.iloc[-1], ends.iloc[0]
ratio = best["median_salary"] / worst["median_salary"]
ax.set_title(
    f"A '{best['skill']}' title pays {ratio:.1f}x what a '{worst['skill']}' title pays",
    fontsize=13.5, loc="left",
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGS / "eda_skill_premium.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Part 6 — From analysis to product: the Career Fit Score

The four things a job seeker needs to weigh are measured in four different units —
counts, dollars, a ratio and a percentage. To combine them we convert each to a
**percentile rank (0–100)** across all 215 career tracks, then take a weighted average.
The weights are sliders in the dashboard, not constants in the code.

In [ ]:
tracks.head(10)[[
    "track", "postings", "median_salary", "apps_per_vacancy",
    "demand_score", "pay_score", "low_competition_score",
    "accessibility_score_default", "fit_score_default",
]].round(1)

**Reading the top of the table.** *Information Technology – Mid* wins on demand and pay
and merely average competition. *Sales / Retail – Junior* wins the opposite way: modest
pay, but 98% of its postings are open to someone with 3 years of experience and only
1.7 applicants per seat. **These are genuinely different recommendations for genuinely
different people** — which is why the score is user-weighted rather than fixed.

In [ ]:
# Sensitivity check: does the ranking actually respond to the weights?
def rank_with(weights):
    w = np.array(weights, dtype=float)
    w = w / w.sum()
    cols = tracks[["demand_score", "pay_score", "low_competition_score",
                   "accessibility_score_default"]].fillna(50).to_numpy()
    scored = tracks.assign(score=cols @ w)
    return scored.nlargest(5, "score")["track"].tolist()

comparison = pd.DataFrame({
    "Balanced (default)": rank_with([30, 30, 25, 15]),
    "Money first": rank_with([10, 70, 10, 10]),
    "Easiest to enter": rank_with([15, 5, 30, 50]),
})
comparison.index = [f"#{i + 1}" for i in range(5)]
comparison

Swapping the weights genuinely swaps the advice. *Money first* keeps the IT tracks but
reorders them and pulls in Banking Management; *Easiest to enter* shares **no entries at
all** with the default — it surfaces HR, Sales and Education entry roles instead. The
score responds to what the user says they care about rather than dressing up one fixed
ranking.

---
## Part 7 — How is the market structured? (Lessons 1.9 and 1.10)

Parts 4–6 looked at 43 categories one at a time. A coach usually starts one level up: *which part of
the economy, and at what level?* This part adds three lenses, adapted from the public
[SG Job Market Trend Analysis dashboard](https://huggingface.co/spaces/Miqqie/job) on Hugging Face
and rebuilt here in pandas on our cleaned data:

| Lens | New column | Rule (in `data_utils.py`) |
|---|---|---|
| Broad sector | `broad_sector` | 43 categories grouped into 7 sectors (`BROAD_SECTORS`) |
| PMET | `pmet` | Only `Non-executive` postings are Non-PMET |
| Listing friction | `is_repost` | An advert put up more than once (`metadata_repostCount > 0`) |

One deliberate difference from that dashboard: it divides applications and views by every posting.
We keep our Part 2.4 rule and use only adverts first posted up to June 2023, because after that
the counters stop building up and competition would look far lower than it is.

> ✍️ The numbers below were not available when this section was written. After *Run All*,
> replace each *Finding* line with one sentence stating what the table or chart shows.

### 7.1 Seven broad sectors

*Question: where are the seats, and how does pay and PMET mix differ by sector?*

In [ ]:
# The notebook's `jobs` table uses friendlier column names; data_utils expects the originals.
clean = jobs.rename(columns={"vacancies": "numberOfVacancies", "company": "postedCompany_name",
                             "avg_salary": "average_salary"})

by_sector = clean.groupby("broad_sector").agg(
    postings=("title_key", "size"),
    vacancies=("numberOfVacancies", "sum"),
    median_salary=("average_salary", "median"),
    pmet_share=("pmet", lambda s: (s == "PMET").mean()),
    repost_share=("is_repost", "mean"),
).sort_values("vacancies", ascending=False)
by_sector["vacancy_share"] = by_sector["vacancies"] / by_sector["vacancies"].sum()
by_sector.round(3)

**Reading it.** `groupby` collapses ~1M rows into 7 (Lesson 1.9 split-apply-combine). This uses each
posting's *first* category, so the shares add up to 100%. Beat 4 check: `by_sector['postings'].sum()`
should equal `len(jobs)`.

*Finding:* …

In [ ]:
assert by_sector["postings"].sum() == len(jobs), "grouped total does not tie back to the row count"
print("Totals tie back:", f"{by_sector['postings'].sum():,} postings")

### 7.2 PMET and non-PMET demand over time

*Question: is demand shifting between professional and non-professional roles?*

In [ ]:
quarters = du.period_summary(clean, "Quarterly")
quarters = quarters[quarters["period"] >= du.TREND_CUTOFF.to_period("Q").start_time]
vac = quarters.pivot_table(index="period", columns="pmet", values="vacancies", aggfunc="sum").fillna(0)
share = vac.div(vac.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 4))
labels = vac.index.to_period("Q").astype(str)
ax.bar(labels, vac.get("PMET", 0), color="#2F6FB0", label="PMET")
ax.bar(labels, vac.get("Non-PMET", 0), bottom=vac.get("PMET", 0), color="#E0B07A", label="Non-PMET")
for x, (total, pmet_pct) in enumerate(zip(vac.sum(axis=1), share.get("PMET", pd.Series(0, index=vac.index)))):
    ax.text(x, total * 1.01, f"{pmet_pct:.0f}% PMET", ha="center", fontsize=9, color="#3d4852")
ax.set_ylabel("Vacancies")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Vacancies per quarter, split by PMET status", fontsize=13, loc="left")  # replace with the finding
fig.tight_layout()
fig.savefig(FIGS / "eda_pmet_quarterly.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it.** Bars are stacked so the total height is all vacancies; the label shows the PMET share.
Read the *share*, not the height: total volume falls later in the series for reasons we have not
confirmed (Part 2.4).

*Finding:* …

### 7.3 Reward against barrier

*Question: in the most crowded categories, what do you get (pay) for what you must bring (experience)?*

In [ ]:
rb = du.reward_barrier(clean, top_n=10, min_postings=2000).sort_values("apps_per_vacancy")
labels = rb["category"] + "  (" + rb["apps_per_vacancy"].round(1).astype(str) + ")"

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
left.barh(labels, rb["median_salary"], color="#2F6FB0")
left.invert_xaxis()
left.set_title("Reward: median salary (S$)", loc="right", fontsize=11)
right.barh(labels, rb["avg_experience"], color="#E07A2E")
right.set_title("Barrier: average years of experience asked", loc="left", fontsize=11)
for ax in (left, right):
    ax.spines[["top", "right", "left"]].set_visible(False)
fig.suptitle("The 10 most competitive categories (applications per vacancy in brackets)",
             fontsize=13, x=0.01, ha="left")  # replace with the finding
fig.tight_layout()
fig.savefig(FIGS / "eda_reward_barrier.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it.** Competition uses adverts first posted up to June 2023. A category that is crowded *and*
asks for little experience is easy to enter but hard to win; one that is crowded but asks for a lot
of experience suggests a queue of experienced applicants.

*Finding:* …

### 7.4 Friction against pay premium

*Question: among the biggest categories, which pay above their sector, and which keep reposting adverts?*

Part 4 found that reposted jobs pay slightly *less* overall. This chart asks the same question category by
category, comparing each with its own broad sector rather than the whole market.

In [ ]:
fp = du.friction_premium(clean, top_n=10)
x, y = fp["salary_premium_pct"], fp["repost_pct"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, s=fp["apps_per_vacancy"].fillna(1) * 60, c=fp["median_salary"], cmap="Blues",
           edgecolor="#2c3e50", linewidth=0.8, alpha=0.85)
for _, r in fp.iterrows():
    ax.annotate(r["category"], (r["salary_premium_pct"], r["repost_pct"]),
                textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.margins(0.15)  # room for the corner labels
ax.axvline(0, color="#8c9196", linestyle="--", linewidth=1)
ax.axhline(y.median(), color="#8c9196", linestyle="--", linewidth=1)
for xa, ya, ha, va, text in [(0.98, 0.98, "right", "top", "pays more, reposted more"),
                             (0.02, 0.98, "left", "top", "pays less, reposted more"),
                             (0.98, 0.02, "right", "bottom", "pays more, reposted less"),
                             (0.02, 0.02, "left", "bottom", "pays less, reposted less")]:
    ax.text(xa, ya, text, transform=ax.transAxes, ha=ha, va=va, fontsize=9.5, color="#5d6d7e")
ax.set_xlabel("Median salary vs its broad sector's median (%)")
ax.set_ylabel("Adverts reposted (%)")
ax.set_title("Pay premium and listing friction in the 10 highest-volume categories",
             fontsize=13, loc="left")  # replace with the finding
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGS / "eda_friction_premium.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading it.** Bubble size is applications per vacancy; colour is median pay. Reposting is a *hint* that a
role is hard to fill, not proof: an employer may repost for other reasons. Treat the corners as questions
to ask a coach, not conclusions.

*Finding:* …

### 7.5 The benchmark table behind the dashboard's *Sector table* tab

In [ ]:
table = du.sector_summary(clean)
table[["category", "broad_sector", "vacancies", "vacancy_share_pct", "apps_per_vacancy",
       "median_salary", "salary_vs_sector", "avg_experience", "experience_vs_sector",
       "repost_pct", "top_titles"]].head(15).round(2)

---
## Summary — what the EDA decided

| Finding | What it changed in the product |
|---|---|
| `occupationId` 100% null; 3,988 wholly empty rows | dropped column, dropped rows |
| 10,000 impossible salaries; 19,647 extreme values | blank the field / cap at 1st–99th percentile, never delete the posting |
| Zero genuine duplicate job ids | de-duplication kept as a safeguard, documented as a no-op |
| A job carries 1.69 categories | two tables; category shares overlap and we say so |
| **Engagement counters die after Jun 2023** | **all competition metrics restricted to Oct 2022 – Jun 2023 and labelled** |
| Volume ramp before May 2023 | trend and growth views start May 2023 |
| Size ≠ pay ≠ low competition | the recommender scores four dimensions instead of ranking on one |
| Reposted jobs pay *less* | "hard to fill" framed as negotiation leverage, not prestige |
| Top posters are recruitment agencies | labelled "poster", never "employer" |
| ISO dates (2024-01-05) were being read day-first (1 May) | dates parsed as year-month-day first |
| 43 categories are too many to start a conversation | 7 broad sectors, PMET and repost filters in the app |
| Pay must be judged against the right peers | Sector table and friction map benchmark each category against its broad sector |

**Next:** `app.py` — the dashboard built on these decisions (`streamlit run app.py`).